## Step 1

Hitting endpoint: https://catalog.redhat.com/api/containers/v1/repositories/registry/registry.access.redhat.com/repository/rhbk%2Fkeycloak-rhel9/images

### Return:

a json with list of all the image metadata and data in the specified Repository (rhbk/keycloak-rhel9)

#### Next Step:

The next step will be analyzing the retrieved images to identify key fields that can be used to group images by their content stream

In [ ]:
import requests
from datetime import datetime

REDHAT_KEYCLOAK_IMAGES_URL = "https://catalog.redhat.com/api/containers/v1/repositories/registry/registry.access.redhat.com/repository/rhbk%2Fkeycloak-rhel9/images"


def fetch_keycloak_images_in_catalog():
    request_headers = {"accept": "application/json"}
    request_response = requests.get(REDHAT_KEYCLOAK_IMAGES_URL, headers=request_headers)
    request_response.raise_for_status()
    return request_response.json()


## Step 2

Filter the returned object from fetch_keycloak_images_in_catalog() function as per Content streams
test endpoint https://catalog.redhat.com/api/containers/v1/repositories/registry/registry.access.redhat.com/repository/rhbk%2Fkeycloak-rhel9
to get Content Stream Tags.

### Return

the same list passed as argument but grouped according to the content stream found under label > version > value

#### Next

in each content stream find the recent image by comparing fields like published_date, last_update_date, creation_date and return only 1 image(recent) per content stream

In [9]:
def group_images_by_version_label(api_response_data):
    images_grouped_by_version = {}

    for image_entry in api_response_data.get("data", []):
        parsed_data_labels = image_entry.get("parsed_data", {}).get("labels", [])
        version_label_value = None

        for label_entry in parsed_data_labels:
            if label_entry.get("name") == "version":
                version_label_value = label_entry.get("value")
                break

        if version_label_value:
            if version_label_value not in images_grouped_by_version:
                images_grouped_by_version[version_label_value] = []
            images_grouped_by_version[version_label_value].append(image_entry)

    return images_grouped_by_version

## Step 3:

Selecting the Most Recent Image per Content Stream, This function processes the grouped image data to identify the most recent image entry within each content stream (version group). It operates on the output of the previous grouping step, which organizes all images by their respective version labels.

### Return

Returns a dictionary mapping each content stream (version) to its latest image entry.

#### Next

add a feature to properly display or return the contents as asked in the problem statement.
required fields and proper formating in JSON.

In [12]:
def select_latest_image_per_version(images_grouped_by_version):
    latest_image_per_version = {}

    for version_label, images_for_version in images_grouped_by_version.items():
        most_recent_image = None
        most_recent_published_date = None

        for image_entry in images_for_version:
            image_repositories = image_entry.get("repositories", [])

            if not image_repositories or not image_repositories[0].get("published_date"):
                continue

            published_date_string = image_repositories[0]["published_date"]
            normalized_date_string = published_date_string.replace("Z","+00:00")
            current_published_date = datetime.fromisoformat(normalized_date_string)

            if most_recent_published_date is None or current_published_date > most_recent_published_date:
                most_recent_published_date = current_published_date
                most_recent_image = image_entry

        if most_recent_image:
            latest_image_per_version[version_label] = most_recent_image

    print(latest_image_per_version)
    return latest_image_per_version

In [13]:
sample = fetch_keycloak_images_in_catalog()
sample_grouped = group_images_by_version_label(sample)
select_latest_image_per_version(sample_grouped)

{'22': {'_id': '66e965a46e919237df246695', 'architecture': 'ppc64le', 'brew': {'build': 'keycloak-rhel9-container-22-18', 'completion_date': '2024-09-17T11:17:26.489000+00:00', 'nvra': 'keycloak-rhel9-container-22-18.ppc64le', 'package': 'keycloak-rhel9-container'}, 'certified': False, 'container_grades': {'status': 'completed', 'status_message': 'This image has unapplied Critical or Important security errata, check the vulnerabilities'}, 'content_sets': ['rhel-9-for-ppc64le-baseos-rpms', 'rhel-9-for-ppc64le-appstream-rpms'], 'cpe_ids': ['cpe:/a:redhat:enterprise_linux:9::appstream', 'cpe:/o:redhat:enterprise_linux:9::baseos'], 'creation_date': '2024-09-17T11:19:00.990000+00:00', 'docker_image_id': 'sha256:85cd0fa99e12322bd37d97f2e9eb251618bc6d4e22a4517e3e65d454b907ceac', 'freshness_grades': [{'creation_date': '2025-10-19T15:13:21.699000+00:00', 'end_date': '2026-07-15T23:11:01+00:00', 'grade': 'D', 'start_date': '2025-10-19T15:13:21.699000+00:00'}, {'creation_date': '2025-10-19T15:13:

{'22': {'_id': '66e965a46e919237df246695',
  'architecture': 'ppc64le',
  'brew': {'build': 'keycloak-rhel9-container-22-18',
   'completion_date': '2024-09-17T11:17:26.489000+00:00',
   'nvra': 'keycloak-rhel9-container-22-18.ppc64le',
   'package': 'keycloak-rhel9-container'},
  'certified': False,
  'container_grades': {'status': 'completed',
   'status_message': 'This image has unapplied Critical or Important security errata, check the vulnerabilities'},
  'content_sets': ['rhel-9-for-ppc64le-baseos-rpms',
   'rhel-9-for-ppc64le-appstream-rpms'],
  'cpe_ids': ['cpe:/a:redhat:enterprise_linux:9::appstream',
   'cpe:/o:redhat:enterprise_linux:9::baseos'],
  'creation_date': '2024-09-17T11:19:00.990000+00:00',
  'docker_image_id': 'sha256:85cd0fa99e12322bd37d97f2e9eb251618bc6d4e22a4517e3e65d454b907ceac',
  'freshness_grades': [{'creation_date': '2025-10-19T15:13:21.699000+00:00',
    'end_date': '2026-07-15T23:11:01+00:00',
    'grade': 'D',
    'start_date': '2025-10-19T15:13:21.6990